In [1]:
import os
import copy
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, Subset, ConcatDataset
from torchvision import datasets
import torchvision.transforms as transforms

import timm
from timm.models.vision_transformer import VisionTransformer

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [2]:
def create_mae_model_from_timm(new_depth=6):
    """Create a ViT-B/16 encoder and load local weights.

    - Avoids any download by setting `pretrained=False`.
    - Loads a local checkpoint if present (keys with optional `module.` prefix).
    """
    encoder = timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,
        img_size=224,
        patch_size=16,
        embed_dim=768,
        depth=new_depth,
        num_heads=12,
        mlp_ratio=4.0
    )

    ckpt_path = '../models/vit_base_patch16_224.pth'
    state_dict = torch.load(ckpt_path, map_location='cpu')

    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

    missing_keys, unexpected_keys = encoder.load_state_dict(state_dict, strict=False)
    print("✅ Loaded encoder weights.")
    print(f"Missing keys: {len(missing_keys)} | Unexpected keys: {len(unexpected_keys)}")

    return encoder


class SimpleMAE(torch.nn.Module):
    """Minimal MAE composed of a ViT encoder + lightweight decoder.

    - `forward(imgs)` returns `(loss, pred_tokens, mask, ids_restore)`.
    - `mask_ratio` controls the fraction of patches masked during encoding.
    """
    def __init__(
        self,
        encoder,
        decoder_embed_dim=512,
        decoder_depth=8,
        decoder_num_heads=16,
        mask_ratio=0.75,
        norm_pix_loss=True
    ):
        super().__init__()
        self.encoder = encoder
        self.img_size = encoder.patch_embed.img_size[0]
        self.patch_size = encoder.patch_embed.patch_size[0]
        self.embed_dim = encoder.embed_dim
        self.num_patches = (self.img_size // self.patch_size) ** 2
        self.mask_token = torch.nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_embed = torch.nn.Linear(self.embed_dim, decoder_embed_dim, bias=True)
        self.decoder_pos_embed = torch.nn.Parameter(torch.zeros(1, self.num_patches + 1, decoder_embed_dim))
        self.decoder_blocks = torch.nn.ModuleList([
            timm.models.vision_transformer.Block(
                decoder_embed_dim, decoder_num_heads, 4.0
            ) for _ in range(decoder_depth)
        ])

        self.decoder_norm = torch.nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = torch.nn.Linear(
            decoder_embed_dim, self.patch_size**2 * 3, bias=True
        )

        self.mask_ratio = mask_ratio
        self.norm_pix_loss = norm_pix_loss
        self.initialize_weights()

    def initialize_weights(self):
        torch.nn.init.normal_(self.mask_token, std=0.02)
        torch.nn.init.normal_(self.decoder_pos_embed, std=0.02)
        torch.nn.init.xavier_uniform_(self.decoder_embed.weight)
        torch.nn.init.zeros_(self.decoder_embed.bias)
        torch.nn.init.xavier_uniform_(self.decoder_pred.weight)
        torch.nn.init.zeros_(self.decoder_pred.bias)

    def random_masking(self, x, mask_ratio):
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(
            x, dim=1,
            index=ids_keep.unsqueeze(-1).repeat(1, 1, D)
        )
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore

    def forward_encoder(self, x, mask_ratio):
        patches = self.encoder.patch_embed(x)
        patches = patches + self.encoder.pos_embed[:, 1:, :]
        patches_masked, mask, ids_restore = self.random_masking(patches, mask_ratio)
        cls_token = self.encoder.cls_token + self.encoder.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(patches_masked.shape[0], -1, -1)
        x = torch.cat((cls_tokens, patches_masked), dim=1)
        for blk in self.encoder.blocks:
            x = blk(x)
        x = self.encoder.norm(x)

        return x, mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        x = self.decoder_embed(x)
        mask_tokens = self.mask_token.repeat(
            x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1
        )
        x_ = x[:, 1:, :]
        x_ = torch.cat([x_, mask_tokens], dim=1)
        x_ = torch.gather(
            x_, dim=1,
            index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2])
        )
        x = torch.cat([x[:, :1, :], x_], dim=1)
        x = x + self.decoder_pos_embed
        for blk in self.decoder_blocks:
            x = blk(x)
        x = self.decoder_norm(x)
        x = self.decoder_pred(x)
        x = x[:, 1:, :]

        return x

    def patchify(self, imgs):
        p = self.patch_size
        h = w = self.img_size // p
        x = imgs.reshape(shape=(imgs.shape[0], 3, h, p, w, p))
        x = torch.einsum('nchpwq->nhwpqc', x)
        x = x.reshape(shape=(imgs.shape[0], h * w, p**2 * 3))

        return x

    def unpatchify(self, x):
        p = self.patch_size
        h = w = int(x.shape[1]**0.5)
        assert h * w == x.shape[1]
        x = x.reshape(shape=(x.shape[0], h, w, p, p, 3))
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(shape=(x.shape[0], 3, h * p, w * p))

        return imgs

    def forward_loss(self, imgs, pred, mask):
        target = self.patchify(imgs)
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1.e-6)**0.5
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()

        return loss

    def forward(self, imgs, mask_ratio=None):
        if mask_ratio is None:
            mask_ratio = self.mask_ratio
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(imgs, pred, mask)
        return loss, pred, mask, ids_restore

In [3]:
class FERClassifier(nn.Module):
    """Simple classifier head on top of a ViT-based MAE encoder.

    Uses the `[CLS]` token from the encoder as the global image representation.
    """
    def __init__(self, mae_encoder, num_classes=7):
        super().__init__()
        self.encoder = mae_encoder
        self.classifier = nn.Sequential(
            nn.LayerNorm(mae_encoder.embed_dim),
            nn.Linear(mae_encoder.embed_dim, num_classes)
        )

    def forward(self, x):
        tokens = self.encoder.patch_embed(x)
        cls_token = self.encoder.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls_token, tokens), dim=1)
        x = x + self.encoder.pos_embed[:, :x.size(1), :]
        x = self.encoder.pos_drop(x)

        for blk in self.encoder.blocks:
            x = blk(x)

        x = self.encoder.norm(x)
        cls_embedding = x[:, 0]
        return self.classifier(cls_embedding)

In [4]:
encoder = create_mae_model_from_timm(new_depth=6)
model = SimpleMAE(encoder)
model.load_state_dict(torch.load('../models/model_epoch_35.pth', map_location='cpu'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model setup
encoder = model.encoder  # Your pretrained MAE encoder
fer_model = FERClassifier(mae_encoder=encoder, num_classes=7).to(device)

✅ Loaded encoder weights.
Missing keys: 0 | Unexpected keys: 72


In [5]:
test_dir = "../datasets/FER_2013_enhanced/preprocessed_test"
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # deterministic resize for val/test
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)
desired_class_order = [
    "neutral",   # 0
    "happy",     # 1
    "sad",       # 2
    "surprise",  # 3
    "fear",      # 4
    "disgust",   # 5
    "anger"      # 6
]
original_class_order = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
label_remap = {
    original_class_order.index('neutral'): desired_class_order.index('neutral'),
    original_class_order.index('happy'): desired_class_order.index('happy'),
    original_class_order.index('sad'): desired_class_order.index('sad'),
    original_class_order.index('surprise'): desired_class_order.index('surprise'),
    original_class_order.index('fear'): desired_class_order.index('fear'),
    original_class_order.index('disgust'): desired_class_order.index('disgust'),
    original_class_order.index('angry'): desired_class_order.index('anger')
}
def remap_labels(dataset):
    """Return a new dataset with labels remapped to `desired_class_order`."""
    # Create new list of samples with remapped labels
    new_samples = []
    for path, label in dataset.samples:
        new_label = label_remap[label]
        new_samples.append((path, new_label))

    # Create new dataset with remapped labels
    new_dataset = datasets.ImageFolder(root=dataset.root, transform=dataset.transform)
    new_dataset.samples = new_samples
    new_dataset.targets = [s[1] for s in new_samples]
    new_dataset.classes = desired_class_order
    new_dataset.class_to_idx = {cls: i for i, cls in enumerate(desired_class_order)}
    return new_dataset
test_dataset = remap_labels(test_dataset)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=os.cpu_count() or 4, pin_memory=(device.type=='cuda'), persistent_workers=True)

In [6]:
criterion = nn.CrossEntropyLoss(reduction='sum')
results = []
fer_model.to(device)

for i in range(1, 26):
    ckpt_path = f"../checkouts/fer_test/fer_model___epoch_{i}.pth"
    if not os.path.exists(ckpt_path):
        print(f"[epoch {i:02d}] checkpoint not found: {ckpt_path}")
        continue

    checkpoint = torch.load(ckpt_path, map_location=device)
    state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    fer_model.load_state_dict(state_dict)
    fer_model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = fer_model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total if total > 0 else float('nan')
    acc = correct / total if total > 0 else float('nan')
    results.append((i, avg_loss, acc))
    print(f"Epoch {i:02d} -> loss: {avg_loss:.4f}, accuracy: {acc*100:.2f}%")


Epoch 01 -> loss: 1.2022, accuracy: 56.78%
Epoch 02 -> loss: 1.1891, accuracy: 57.19%
Epoch 03 -> loss: 1.1833, accuracy: 57.19%
Epoch 04 -> loss: 1.1758, accuracy: 57.43%
Epoch 05 -> loss: 1.1742, accuracy: 57.43%
Epoch 06 -> loss: 1.1647, accuracy: 57.72%
Epoch 07 -> loss: 0.9744, accuracy: 63.88%
Epoch 08 -> loss: 0.9553, accuracy: 64.98%
Epoch 09 -> loss: 0.8604, accuracy: 67.93%
Epoch 10 -> loss: 0.8653, accuracy: 67.86%
Epoch 11 -> loss: 0.8476, accuracy: 68.31%
Epoch 12 -> loss: 0.8416, accuracy: 68.97%
Epoch 13 -> loss: 0.8390, accuracy: 70.03%
Epoch 14 -> loss: 0.8381, accuracy: 69.18%
Epoch 15 -> loss: 0.8508, accuracy: 69.62%
Epoch 16 -> loss: 0.8427, accuracy: 69.74%
Epoch 17 -> loss: 0.8359, accuracy: 70.49%
Epoch 18 -> loss: 0.8457, accuracy: 70.23%
Epoch 19 -> loss: 0.8493, accuracy: 70.14%
Epoch 20 -> loss: 0.8816, accuracy: 69.85%
Epoch 21 -> loss: 0.8634, accuracy: 70.26%
Epoch 22 -> loss: 0.8788, accuracy: 69.99%
Epoch 23 -> loss: 0.8803, accuracy: 70.76%
Epoch 24 ->